<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>


# ROS 2 Node Lifecycles and Publishers

A program that shares readings or status needs to produce messages periodically while remaining available to the graph. `rclpy` provides nodes, publishers, and timers for that work. We will follow the starter's process lifecycle, run it, and change its payload and rate to see how settings control the publish callback.

## Understand the node lifecycle

The starter follows the process lifecycle in [Figure 1](#figure-1). Initialization prepares the client library; spinning keeps callbacks running; cleanup releases the node's resources.

<figure id="figure-1" class="lx-figure" style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    `rclpy.init()`
      |
      v
    create a Node
      |
      v
    `rclpy.spin(node)`
      |
      v
    callbacks run
      |
      v
    `destroy_node()`
      |
      v
    `rclpy.shutdown()`
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">
    Figure 1: Lifecycle of the ROS 2 publisher starter.
  </figcaption>
</figure>

`rclpy.init()` initializes the library for this process. The starter's node constructor creates its publisher and timer. `rclpy.spin(node)` waits for work and lets the executor call the timer callback. When you stop the program with `Ctrl-C`, cleanup destroys the node and shuts down `rclpy`.

## Read the publisher starter

Open [packages/ros2_pubsub/ros2_pubsub/my_publisher.py](../packages/ros2_pubsub/ros2_pubsub/my_publisher.py). This working first version lets you connect the settings to observable output before changing them. Keep `MSG_TYPE = String`. The initial configuration is:

```python
MSG_TYPE = String
PUBLISH_RATE_HZ = 1.0
MESSAGE_CONTENT = "Hello from ROS 2!"
TOPIC_NAME = "chatter"
NODE_NAME = "talker"
QUEUE_DEPTH = 10
```

`String` stores text in its `data` field. The `Talker` node creates a publisher and a timer. Each timer callback creates a message, sets `data`, logs the text, and publishes on `TOPIC_NAME`.

The rate is measured in hertz: `1.0` means one message each second. The starter converts it to the timer period in seconds. `QUEUE_DEPTH = 10` sets the depth of the recent-message history. These QoS settings suit the local exercise, while sensor subscriptions may need different policies.

## Build and run the publisher

From the supplied ROS 2 exercise terminal, build the current source and source the overlay so the command runs that installed version:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
cd "$COLCON_WS"
colcon build --packages-select ros2_pubsub
source install/setup.bash
ros2 run ros2_pubsub my_publisher
```

With the initial settings, repeated lines containing `talker` and `Hello from ROS 2!` show that the publish callback is running with the configured text. At `1.0` hertz, expect about one line each second. Timestamps and log formatting can vary. Stop the node with `Ctrl-C` before editing.

## Change the payload and publication rate

Change `MESSAGE_CONTENT` to your own text. Rebuild, source the overlay, and run again:

```bash
cd "$COLCON_WS"
colcon build --packages-select ros2_pubsub
source install/setup.bash
ros2 run ros2_pubsub my_publisher
```

Seeing the new text connects the source change to the installed node you are running. Next, set `PUBLISH_RATE_HZ = 2.0` and repeat the sequence; about two log lines each second demonstrate the timer's new schedule. Keep the rate positive because the timer period is computed from it, and keep `MSG_TYPE = String`.

Use [Notebook 7](./7-inspect-a-local-ros-2-graph.ipynb) to observe the published payload and rate from a separate subscription. This connects the callback's local log to messages received through the graph.

## Further reading

The ROS 2 Jazzy tutorial on [writing a simple Python publisher and subscriber](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Writing-A-Simple-Py-Publisher-And-Subscriber.html) provides a second worked example.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
